"""
# 1- get a train_directory path
# 2- read the json, and the pred tensors
# 3- recreate the training environment with the args
# 5- get the shared_df from the inference_val_dataset, get the initial_price and the start date from it
# 6- re-scale the predicted prices
# 7- plot the price line
"""

In [ ]:
import os
import json
import importlib
import inspect
import argparse
import pandas as pd

from tqdm import tqdm
import matplotlib.pyplot as plt
from collections import OrderedDict

import numpy as np
import torch
from torch.utils.data import DataLoader

from import_model import import_model
from import_dataset import import_dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
train_session_dir = "train_sessions/EncoderDecoderLSTM_20250625_212043"

In [ ]:
json_to_inference = os.path.join(train_session_dir,"args.json")
with open(json_to_inference, 'r') as f:
    data = json.load(f)

In [ ]:
model_name = data["model_name"]

train_target_name = f'{model_name}_train_target_series.pt'
train_pred_name = f'{model_name}_train_pred_series.pt'
val_target_name = f'{model_name}_val_target_series.pt'
val_pred_name = f'{model_name}_val_pred_series.pt'

train_target_path = os.path.join(train_session_dir, train_target_name)
train_pred_path = os.path.join(train_session_dir, train_pred_name)
val_target_path = os.path.join(train_session_dir, val_target_name)
val_pred_path = os.path.join(train_session_dir, val_pred_name)

target_and_pred_names = [train_target_name, train_pred_name, val_target_name, val_pred_name]
model_pt_name = [x for x in os.listdir(train_session_dir) if x.endswith(".pt") and x not in target_and_pred_names][0]
model_pt_path = os.path.join(train_session_dir, model_pt_name)

In [ ]:
args = argparse.Namespace(**data)

base_dataset_kwargs = { "coin_symbol": args.coin_symbol, "input_window": args.input_window, "output_window": args.output_window,
    "augmentation_noise_std": args.augmentation_noise_std, "augment_constant_c": args.augment_constant_c, "augment_scale_s": args.augment_scale_s,
    "z_norm_means_csv_path": args.z_norm_means_csv_path, "z_norm_stds_csv_path": args.z_norm_stds_csv_path,
    "distribution_scale": args.distribution_scale, "distribution_clip": args.distribution_clip, "transform_name":args.transform_name,
    "output_distribution": args.output_distribution, "n_quantiles": args.n_quantiles, "train_session_dir": train_session_dir}

inference_val_dataset_kwargs = {**base_dataset_kwargs, "csv_path": "../data/BTC_ETH_BNB_XRP_6h_log_returns_val.csv", "augmentation_p": 0, "load_transform":1}
inference_val_dataset = import_dataset(args.dataset_name, **inference_val_dataset_kwargs)
inference_val_loader = DataLoader(inference_val_dataset, batch_size=args.batch_size, shuffle=False)

model_kwargs = {"input_features": args.input_features, "output_features": args.output_features,
"input_window": args.input_window, "output_window": args.output_window,
"dropout": args.dropout, "num_layers": args.num_layers,
"hidden_dim": args.hidden_dim, "num_heads": args.num_heads,
"teacher_forcing_ratio": args.teacher_forcing_ratio,
"target_coin_index": args.target_coin_index,
"num_coins": args.num_coins, "device": device}

In [ ]:
model = import_model(args.model_name, **model_kwargs)
model.load_state_dict(torch.load(model_pt_path, weights_only=True))

In [ ]:
train_target_series = torch.load(train_target_path)
print("train_target_series:", train_target_series.shape)
train_pred_series = torch.load(train_pred_path)
print("train_pred_series:", train_pred_series.shape)
val_target_series = torch.load(val_target_path)
print("val_target_series:", val_target_series.shape)
val_pred_series = torch.load(val_pred_path)
print("val_pred_series:", val_pred_series.shape)

In [ ]:
# price = val_target_series[:, 2, 0].unsqueeze(0)
# y_pred_full = np.zeros((price.shape[0], 16))
# y_pred_full[:, :4] = price
# y_pred_inversed = inference_val_dataset.transform.inverse_transform(y_pred_full)
# y_pred_rescaled = y_pred_inversed[:, :4]
# y_pred_rescaled

In [ ]:
inference_val_dataset.df

In [ ]:
trust = 0
predicted_dataframe_crop = inference_val_dataset.df.iloc[data["input_window"]+1+trust:-(data["output_window"]-2-trust)]
predicted_dataframe_crop

In [ ]:
[data["input_window"]+1+trust:-(data["output_window"]-2-trust)]

In [ ]:
df = pd.read_csv("/home/mericdemirors/Desktop/lecture slides/TUD_lectures/S2/deep_learning_architectures_and_methods/CrossCurrencyPrediction/data/BTC_ETH_BNB_XRP_6h_shared.csv")
val_df = df[-len(inference_val_dataset.df):]
val_df_preds = val_df.iloc[data["input_window"]:-data["output_window"]+1]
val_df.tail(10)

In [ ]:
val_df_pr

In [ ]:
target_series_to_plot = torch.cat((val_target_series[:,:,0], val_target_series[:,-1]), dim=1)
pred_series_with_different_trusts = []
for trust in range(8):
    pred_series_with_different_trusts.append(torch.cat((torch.zeros(val_pred_series.shape[0], trust), val_pred_series[:,:,trust], val_pred_series[:,-1,trust:]), dim=1))

feature_names = ["Open", "Close", "Low", "High"]
plt.figure(figsize=(20, 10))

for i in range(4):
    plt.subplot(2, 2, i + 1)
    
    rescaled_target = inference_val_dataset.rescale_to_real_price(target_series_to_plot[i])
    plt.plot(rescaled_target, label="Ground Truth", color="orange")
    
    for trust, pred_series_to_plot in enumerate(pred_series_with_different_trusts):
        rescaled_pred = inference_val_dataset.rescale_to_real_price(pred_series_to_plot[i])
        plt.plot(rescaled_pred, label="Prediction", color="green", alpha=1/(trust+1))

    plt.title(f'{feature_names[i]}')
    plt.xlabel("Time")
    plt.ylabel("Value")
    plt.legend()

    handles, labels = plt.gca().get_legend_handles_labels()
    by_label = OrderedDict(zip(reversed(labels), reversed(handles)))
    plt.legend(by_label.values(), by_label.keys())    

plt.tight_layout()
plt.show()